In [ ]:
wg_sets = [
    {
        "description": "First spatial lag, group concentric rings",
        "variables": {
            "y": "gva",
            "k": "total_assets",
            "l": "employees",
        },
        "type": "g",
        "transforms": {
            "g1": "pc8",
            "g2": "pc4",
            "g3": "ttwa"
        }
    },
    {
        "description": "All combinations of second-order group spatial lags",
        "variables": [
            "wg1_k", "wg1_l",
            "wg2_k", "wg2_l",
            "wg3_k", "wg3_l"
        ],
        "type": "g",
        "transforms": {
            "g1": "pc8",
            "g2": "pc4",
            "g3": "ttwa"
        }
    },
    {
        "description": "Third and 4th order group lags for wg1",
        "variables": [
            "w2g1_k", "w2g1_l",
            "w3g1_k", "w3g1_l"
        ],
        "type": "g",
        "transforms": {
            "g1": "pc8"
        }
    },
    {
        "description": "Local and global FE transforms for w2g1",
        "variables": [
            "w2g1_k", "w2g1_l",
            "w3g1_k", "w3g1_l"
        ],
        "type": "g",
        "transforms": {
            "(i-wg1)": {
                "i_minus": True,
                "value": "pc8"
            },
            "(i-vg1)": {
                "i_minus": True,
                "leave_one_out": False,
                "value": "pc8"
            }
        }
    }
]

wd_sets = [
    {
        "description": "Distance-weighted spatial lags",
        "variables": {
            "y": "gva",
            "k": "total_assets",
            "l": "employees"
        },
        "type": "n",
        "transforms": [
            "d1",
            "d2",
            "d3"
        ]
    },
    {
        "description": "Higher-index d1 lags",
        "variables": [
            "wd1_k", "wd1_l",
            "w2d1_k", "w2d1_l",
            "w3d1_k", "w3d1_l"
        ],
        "type": "n",
        "transforms": [
            "d1"
        ]
    },
    {
        "description": "Higher-index d2 lags",
        "variables": [
            "wd2_k", "wd2_l"
        ],
        "type": "n",
        "transforms": [
            "d2"
        ]
    },
    {
        "description": "Higher-index d3 lags",
        "variables": [
            "wd3_k", "wd3_l"
        ],
        "type": "n",
        "transforms": [
            "d3"
        ]
    },
    {
        "description": "Local and global FE transforms for w2d1",
        "variables": [
            "w2d1_k", "w2d1_l",
            "w3d1_k", "w3d1_l"
        ],
        "type": "n",
        "transforms": {
            "(i-wd1)": {
                "i_minus": True,
                "value": "d1"
            },
            "(i-vd1)": {
                "i_minus": True,
                "leave_one_out": False,
                "value": "d1"
            }
        }
    }
]

In [ ]:
import re
from collections.abc import Mapping, Sequence
from typing import Any

import ibis
from ibis import _
from utils.f_0_dirs import get_data_dirs


TransformSpec = str | Mapping[str, Any]
VariableSpec = Mapping[str, str] | Sequence[str]
TransformConfig = tuple[str, str, bool, bool]


def _normalise_variables(variables: VariableSpec) -> list[tuple[str, str]]:
    """
    Return (output_base_name, input_column) pairs.

    A mapping is treated as {logical_name: existing_column}; a sequence means
    the existing column name is also the output base name.
    """
    if isinstance(variables, Mapping):
        return list(variables.items())

    if isinstance(variables, str):
        raise TypeError("'variables' must be a mapping or a sequence of column names")

    return [(column, column) for column in variables]


def _normalise_transforms(
    transforms: Mapping[str, TransformSpec] | Sequence[str],
) -> list[tuple[str, str, bool, bool]]:
    """
    Return (transform_name, value, leave_one_out, i_minus) pairs in schema order.

    For mapping values:
      - a string is the group/network column to use;
      - a mapping must contain 'value' and may contain 'leave_one_out'
        and 'i_minus'.

    The insertion order of the supplied mapping is preserved.
    """
    if isinstance(transforms, Mapping):
        items = transforms.items()
    else:
        if isinstance(transforms, str):
            raise TypeError("'transforms' must be a mapping or a sequence of transform names")
        items = ((name, name) for name in transforms)

    normalised: list[TransformConfig] = []

    for name, spec in items:
        if isinstance(spec, Mapping):
            if "value" not in spec:
                raise ValueError(f"Transform {name!r} is missing required key 'value'")

            value = spec["value"]
            leave_one_out = bool(spec.get("leave_one_out", True))
            i_minus = bool(spec.get("i_minus", False))
        else:
            value = spec
            leave_one_out = True
            i_minus = False

        if not isinstance(value, str):
            raise TypeError(
                f"Transform {name!r}: 'value' must identify a table column"
            )

        normalised.append((name, value, leave_one_out, i_minus))

    return normalised


def _make_w_name(input_col: str, transform_name: str, output_base: str | None = None) -> str:
    """
    Construct the column name for W x.

    Naming rules:
      * A plain variable receives ``w<transform>_<variable>``.
      * If the input name already contains ``_``, prefix the transform directly,
        e.g. ``wg1_k`` -> ``wg2wg1_k`` rather than inserting another ``_``.
      * If the immediately preceding transform is the same simple transform,
        increase its order: g1 + wg1_k -> w2g1_k, then w3g1_k, etc.
      * ``output_base`` is used for the initial mapped-variable case, where
        ``{"y": "gva"}`` should produce ``wg1_y`` rather than ``wg1_gva``.
    """
    name = output_base if output_base is not None else input_col

    # Only simple transform names (e.g. g1, d1) get the compact squared notation.
    same_transform = re.match(
        rf"^w(\d*){re.escape(transform_name)}(?=_|$)",
        name,
    )
    if same_transform:
        order = int(same_transform.group(1) or "1") + 1
        return f"w{order}{transform_name}{name[same_transform.end():]}"

    transform_prefix = f"w{transform_name}"

    if "_" in name:
        return f"{transform_prefix}{name}"

    return f"{transform_prefix}_{name}"


def apply_group_W(
    t_panel: ibis.Table,
    input_col: str,
    group_col: str,
    out_col: str,
    *,
    leave_one_out: bool = True,
    i_minus: bool = False,
) -> ibis.Table:
    """
    Apply the row-normalised group matrix W to one existing vector.

    For leave_one_out=True, the target observation is removed:
        Wx_i = (sum_g x_j - x_i) / (n_g - 1)

    For leave_one_out=False, the target observation is retained:
        Wx_i = sum_g x_j / n_g

    If i_minus=True, return x_i - Wx_i, implementing (I - W)x.
    """
    group_sum = t_panel[input_col].sum().over(
        group_by=[t_panel[group_col], t_panel.year]
    )
    group_count = t_panel[input_col].count().over(
        group_by=[t_panel[group_col], t_panel.year]
    )

    if leave_one_out:
        numerator = group_sum - t_panel[input_col]
        denominator = group_count - 1
    else:
        numerator = group_sum
        denominator = group_count

    wx = ibis.ifelse(
        denominator > 0,
        numerator / denominator,
        ibis.NA,
    )

    result = t_panel.mutate(**{out_col: t_panel[input_col] - wx if i_minus else wx})
    return result


def apply_network_W(
    t_panel: ibis.Table,
    t_distance: ibis.Table,
    input_col: str,
    weight_col: str,
    out_col: str,
    *,
    leave_one_out: bool = True,
    i_minus: bool = False,
) -> ibis.Table:
    """
    Apply a distance-weighted network matrix W to one existing vector.

    Standard network case (leave_one_out=True):
        Wx_i = sum_j d_ij x_j / sum_j d_ij

    Leave-one-in / hybrid case (leave_one_out=False):
    the target is treated as a self-observation with unit weight:
        Wx_i = (x_i + sum_j d_ij x_j) / (1 + sum_j d_ij)

    This implements the overlapping, target-specific 'group' implied by the
    distance network while retaining the supplied distance-decay weights.

    If i_minus=True, return x_i - Wx_i, implementing (I - W)x.
    """
    t_target = t_panel.select(
        target_firm=t_panel.registered_number,
        target_year=t_panel.year,
        target_val=t_panel[input_col],
    )

    t_peer = t_panel.select(
        peer_firm=t_panel.registered_number,
        peer_year=t_panel.year,
        peer_val=t_panel[input_col],
    )

    # Build the weighted peer sums only once for this particular input vector
    # and weight definition; all other panel manipulation remains outside this
    # basis-level calculation.
    t_peer_weighted = (
        t_distance
        .inner_join(
            t_target,
            t_distance.firm_i == _.target_firm,
        )
        .inner_join(
            t_peer,
            (t_distance.firm_j == _.peer_firm)
            & (_.target_year == _.peer_year),
        )
        .filter(_.peer_val.notnull())
        .mutate(
            weighted_val=_.peer_val * _[weight_col],
        )
        .group_by("target_firm", "target_year")
        .aggregate(
            weighted_sum=_.weighted_val.sum(),
            weight_sum=_[weight_col].sum(),
        )
    )

    t_result = t_panel.left_join(
        t_peer_weighted,
        (t_panel.registered_number == _.target_firm)
        & (t_panel.year == _.target_year),
    )

    if leave_one_out:
        wx = ibis.ifelse(
            _.weight_sum > 0,
            _.weighted_sum / _.weight_sum,
            ibis.NA,
        )
    else:
        # The target firm is added with unit weight. Coalescing the peer
        # aggregates means an isolated firm correctly has Wx_i = x_i.
        peer_weight_sum = ibis.coalesce(_.weight_sum, ibis.literal(0))
        peer_weighted_sum = ibis.coalesce(_.weighted_sum, ibis.literal(0))
        denominator = peer_weight_sum + 1
        numerator = peer_weighted_sum + t_result[input_col]
        wx = ibis.ifelse(
            denominator > 0,
            numerator / denominator,
            ibis.NA,
        )

    transformed = t_result[input_col] - wx if i_minus else wx

    return t_result.mutate(**{out_col: transformed}).drop(
        ["target_firm", "target_year", "target_val", "weighted_sum", "weight_sum"]
    )


def apply_w_schema(
    t_panel: ibis.Table,
    t_distance: ibis.Table,
    w_sets: Sequence[Mapping[str, Any]],
) -> ibis.Table:
    """
    Apply the complete ordered W-transform schema.

    Each transform is applied independently to the named input variable.
    Higher-order products are obtained by referring to the newly-created
    column in a later `variables` entry.
    """
    t_current = t_panel

    for w_set in w_sets:
        transform_type = w_set["type"]
        variable_specs = _normalise_variables(w_set["variables"])
        transform_specs = _normalise_transforms(w_set["transforms"])

        if transform_type not in {"g", "n"}:
            raise ValueError(
                f"Unsupported W-transform type {transform_type!r}; expected 'g' or 'n'"
            )

        for output_base, input_col in variable_specs:
            if input_col not in t_current.columns:
                raise KeyError(
                    f"Input column {input_col!r} required by the W schema "
                    f"is not present in the current panel"
                )

            # All transforms in this set act on the same input vector. The
            # order of transform evaluation is still preserved exactly.
            for transform_name, value, leave_one_out, i_minus in transform_specs:
                out_col = _make_w_name(
                    input_col,
                    transform_name,
                    output_base=output_base if input_col != output_base else None,
                )

                if out_col in t_current.columns:
                    raise ValueError(
                        f"W-transform would overwrite existing column {out_col!r}"
                    )

                if transform_type == "g":
                    t_current = apply_group_W(
                        t_current,
                        input_col=input_col,
                        group_col=value,
                        out_col=out_col,
                        leave_one_out=leave_one_out,
                        i_minus=i_minus,
                    )
                else:
                    t_current = apply_network_W(
                        t_current,
                        t_distance=t_distance,
                        input_col=input_col,
                        weight_col=value,
                        out_col=out_col,
                        leave_one_out=leave_one_out,
                        i_minus=i_minus,
                    )

    return t_current


# ---------------------------------------------------------------------------
# Common table setup: preserve the existing panel and distance construction.
# ---------------------------------------------------------------------------

panel_name = "working_yearly"
fixed_name = "working_fixed"
distance_name = "working_distance_ttwa_km"

# ---
dirs = get_data_dirs(segment="calculations")
con = ibis.duckdb.connect(dirs.db_path)
dirs = get_data_dirs(segment="calculations")

table_panel = (
    con.table(panel_name)
    .select("registered_number", "year", "gva1", "total_assets", "employees")
    .distinct(on=["registered_number", "year"])
)
table_fixed = (
    con.table(fixed_name)
    .select("registered_number", "pc8", "pc4", "ttwa")
    .distinct(on="registered_number")
)
table_distance = (
    con.table(distance_name)
    .inner_join(table_panel, _.firm_i == table_panel.registered_number)
    .mutate(
        d1=1 / (_.distance_meters + 1),
        d2=1 / (_.distance_meters + 1) ** 2,
        d3=(-_.distance_meters / 1000).exp(),
    )
)
t_fixed_groups = table_fixed.select(
    fixed_firm=table_fixed.registered_number,
    pc8=table_fixed.pc8,
    pc4=table_fixed.pc4,
    ttwa=table_fixed.ttwa,
)
t_current = table_panel
t_current = t_current.left_join(
    t_fixed_groups,
    t_current.registered_number == _.fixed_firm,
).drop("fixed_firm")


# ---------------------------------------------------------------------------
# Execute the schema in order.
# ---------------------------------------------------------------------------

for w_set in (wg_sets, wd_sets):
    t_current = apply_w_schema(t_current, table_distance, w_set)

print(t_current.schema())
print(t_current.sample(0.0001).execute())
